# 02 - Preprocesarea datelor

Acest notebook pregătește datele pentru clasificarea automată a produselor. Se curăță denumirile coloanelor, se elimină valorile lipsă, se normalizează textul și se salvează un fișier curățat care va fi folosit în notebook-ul de antrenare.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA_PATH = Path('products.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/mnt/data/products.csv')

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.head()

## 1. Selectarea coloanelor relevante

Pentru clasificarea automată folosim:
- `Product Title` ca text de intrare;
- `Category Label` ca etichetă / categorie de prezis.

In [ ]:
data = df[['Product Title', 'Category Label']].copy()
data = data.rename(columns={
    'Product Title': 'product_title',
    'Category Label': 'category'
})

data.head()

## 2. Eliminarea valorilor lipsă

Rândurile fără titlu sau fără categorie nu pot fi folosite pentru antrenarea modelului.

In [ ]:
print('Înainte de curățare:', data.shape)
data = data.dropna(subset=['product_title', 'category']).copy()
print('După eliminarea valorilor lipsă:', data.shape)

## 3. Funcție de curățare a textului

Curățarea aplicată:
- transformare în litere mici;
- eliminarea caracterelor speciale;
- eliminarea spațiilor multiple;
- eliminarea spațiilor de la început și final.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

data['clean_title'] = data['product_title'].apply(clean_text)
data['category'] = data['category'].astype(str).str.strip()

data[['product_title', 'clean_title', 'category']].head(10)

## 4. Eliminarea titlurilor goale după curățare

In [ ]:
data = data[data['clean_title'].str.len() > 0].copy()
print('Dimensiune finală:', data.shape)
print('Număr categorii:', data['category'].nunique())

## 5. Împărțirea datelor în train/test

Folosim `stratify=y` pentru a păstra proporția categoriilor în train și test.

In [ ]:
X = data['clean_title']
y = data['category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Train:', X_train.shape)
print('Test:', X_test.shape)

## 6. Salvarea datelor curățate

Fișierul `products_cleaned.csv` va fi folosit în notebook-ul 03 pentru antrenarea și compararea modelelor.

In [ ]:
OUTPUT_PATH = Path('products_cleaned.csv')
data.to_csv(OUTPUT_PATH, index=False)
print(f'Fișier salvat: {OUTPUT_PATH.resolve()}')

## Concluzii

- Datele au fost curățate și standardizate.
- Textul este pregătit pentru vectorizare cu TF-IDF.
- Au fost eliminate rândurile care nu pot fi folosite în clasificare.
- Datele curățate sunt salvate pentru etapa de antrenare.